In [ ]:
# Práctica 1: SparkSQL
## Tarea 3 - Comparación de formatos CSV y Parquet

En esta tarea se continúa trabajando con el conjunto de datos HHAR.

En primer lugar, se convierten los ficheros CSV originales a formato Parquet
para comparar el espacio ocupado por ambos formatos.

Posteriormente, se compara el tiempo de ejecución del procesamiento realizado
en la Tarea 2 utilizando como origen los ficheros Parquet y los ficheros CSV.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Practica1_Tarea3") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark funcionando correctamente")
print("Versión de Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 14:37:14 WARN Utils: Your hostname, MacBook-Air-de-Carmen-2.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
26/11/11 14:37:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 14:37:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 14:37:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark funcionando correctamente
Versión de Spark: 4.2.0


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51258)
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/site-packages/pyspark/accumulators.py", line 329, in handle
    poll(accum_updates)
    ~~~~^^^^^^^^^^^^^^^
  File "/o

## 1. Comparación del tamaño de los ficheros CSV y Parquet

Se crean DataFrames a partir de los cuatro ficheros CSV del dataset HHAR,
proporcionando explícitamente su esquema. Posteriormente, cada DataFrame
se almacena en formato Parquet para comparar el tamaño de ambos formatos.

In [3]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType
)

schema = StructType([
    StructField("Index", LongType(), True),
    StructField("Arrival_Time", LongType(), True),
    StructField("Creation_Time", LongType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", DoubleType(), True),
    StructField("User", StringType(), True),
    StructField("Model", StringType(), True),
    StructField("Device", StringType(), True),
    StructField("gt", StringType(), True)
])

In [4]:
def cargar_csv(ruta):
    """
    Carga un fichero CSV del dataset HHAR como un DataFrame de Spark.

    Parámetros:
        ruta (str): Ruta del fichero CSV.

    Retorna:
        DataFrame: DataFrame creado utilizando el esquema HHAR.
    """
    return spark.read.csv(ruta, schema=schema, header=False)


phones_acc = cargar_csv("Phones_accelerometer.csv")
phones_gyro = cargar_csv("Phones_gyroscope.csv")
watch_acc = cargar_csv("Watch_accelerometer.csv")
watch_gyro = cargar_csv("Watch_gyroscope.csv")

print("DataFrames CSV creados correctamente")

DataFrames CSV creados correctamente


### Conversión de CSV a Parquet

A continuación se almacenan los cuatro DataFrames en formato Parquet.
Este formato utiliza almacenamiento columnar y permite comparar el espacio
ocupado con los ficheros CSV originales.


In [6]:
phones_acc.write.mode("overwrite").parquet("Phones_accelerometer.parquet")
phones_gyro.write.mode("overwrite").parquet("Phones_gyroscope.parquet")
watch_acc.write.mode("overwrite").parquet("Watch_accelerometer.parquet")
watch_gyro.write.mode("overwrite").parquet("Watch_gyroscope.parquet")

print("Ficheros Parquet generados correctamente")

26/11/11 14:38:35 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/11/11 14:39:03 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/11/11 14:39:23 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/11/11 14:39:34 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/11/11 14:39:41 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
[Stage 3:=======>                                                   (1 + 7) / 8]

Ficheros Parquet generados correctamente


### Comparación de tamaños

Se calcula el tamaño ocupado por cada fichero CSV original y por el
directorio Parquet correspondiente para comparar ambos formatos.

In [7]:
import os

def obtener_tamano(ruta):
    """
    Calcula el tamaño total de un fichero o directorio.

    Parámetros:
        ruta (str): Ruta del fichero o directorio.

    Retorna:
        float: Tamaño total expresado en MB.
    """
    if os.path.isfile(ruta):
        tamano = os.path.getsize(ruta)
    else:
        tamano = sum(
            os.path.getsize(os.path.join(carpeta, archivo))
            for carpeta, _, archivos in os.walk(ruta)
            for archivo in archivos
        )

    return tamano / (1024 ** 2)

In [8]:
ficheros = [
    ("Phones_accelerometer", "Phones_accelerometer.csv", "Phones_accelerometer.parquet"),
    ("Phones_gyroscope", "Phones_gyroscope.csv", "Phones_gyroscope.parquet"),
    ("Watch_accelerometer", "Watch_accelerometer.csv", "Watch_accelerometer.parquet"),
    ("Watch_gyroscope", "Watch_gyroscope.csv", "Watch_gyroscope.parquet")
]

resultados_tamano = []

for nombre, csv, parquet in ficheros:
    tamano_csv = obtener_tamano(csv)
    tamano_parquet = obtener_tamano(parquet)

    resultados_tamano.append(
        (nombre, tamano_csv, tamano_parquet)
    )

tabla_tamanos = spark.createDataFrame(
    resultados_tamano,
    ["Fichero", "CSV_MB", "Parquet_MB"]
)

tabla_tamanos.show(truncate=False)

+--------------------+------------------+------------------+
|Fichero             |CSV_MB            |Parquet_MB        |
+--------------------+------------------+------------------+
|Phones_accelerometer|1232.010199546814 |249.89771461486816|
|Phones_gyroscope    |1315.2557916641235|267.39266300201416|
|Watch_accelerometer |312.01176834106445|72.51578140258789 |
|Watch_gyroscope     |294.0531015396118 |64.52797412872314 |
+--------------------+------------------+------------------+



## 2. Comparación de tiempos de ejecución

En este apartado se mide el tiempo necesario para realizar el procesamiento
de la Tarea 2 utilizando dos fuentes de datos diferentes:

- **Caso 1:** DataFrames creados a partir de los ficheros Parquet.
- **Caso 2:** DataFrames creados a partir de los ficheros CSV originales.

En ambos casos se realiza el mismo procesamiento: agregación por `User`,
`Model` y `gt`, cálculo de estadísticas, unión de acelerómetro y giroscopio
mediante `join` y unión final de teléfonos y relojes mediante `union`.

In [9]:
from pyspark.sql import functions as F
import time

def agregar_movimientos(df, prefijo):
    """
    Agrupa los datos por User, Model y gt y calcula las estadísticas
    de los ejes x, y y z.

    Parámetros:
        df (DataFrame): DataFrame con los datos del sensor.
        prefijo (str): Prefijo para identificar el tipo de sensor.

    Retorna:
        DataFrame: DataFrame agregado con media, desviación estándar,
        máximo y mínimo de x, y y z.
    """
    return df.groupBy("User", "Model", "gt").agg(
        F.mean("x").alias(f"{prefijo}_x_media"),
        F.stddev("x").alias(f"{prefijo}_x_std"),
        F.max("x").alias(f"{prefijo}_x_max"),
        F.min("x").alias(f"{prefijo}_x_min"),

        F.mean("y").alias(f"{prefijo}_y_media"),
        F.stddev("y").alias(f"{prefijo}_y_std"),
        F.max("y").alias(f"{prefijo}_y_max"),
        F.min("y").alias(f"{prefijo}_y_min"),

        F.mean("z").alias(f"{prefijo}_z_media"),
        F.stddev("z").alias(f"{prefijo}_z_std"),
        F.max("z").alias(f"{prefijo}_z_max"),
        F.min("z").alias(f"{prefijo}_z_min")
    )

### Caso 1: DataFrames creados a partir de Parquet

Se mide el tiempo necesario para leer los cuatro ficheros Parquet y realizar
el procesamiento completo de la Tarea 2.

Se utiliza la acción `count()` para forzar la ejecución de las transformaciones
y poder medir el tiempo real de procesamiento.

In [10]:
inicio_parquet = time.time()

# Lectura de los ficheros Parquet
p_acc = spark.read.parquet("Phones_accelerometer.parquet")
p_gyro = spark.read.parquet("Phones_gyroscope.parquet")
w_acc = spark.read.parquet("Watch_accelerometer.parquet")
w_gyro = spark.read.parquet("Watch_gyroscope.parquet")

# Agregación
p_acc_agg = agregar_movimientos(p_acc, "acc")
p_gyro_agg = agregar_movimientos(p_gyro, "gyro")
w_acc_agg = agregar_movimientos(w_acc, "acc")
w_gyro_agg = agregar_movimientos(w_gyro, "gyro")

# Join de teléfonos
phones_parquet = p_acc_agg.join(
    p_gyro_agg,
    ["User", "Model", "gt"],
    "inner"
)

# Join de relojes
watch_parquet = w_acc_agg.join(
    w_gyro_agg,
    ["User", "Model", "gt"],
    "inner"
)

# Union final
final_parquet = phones_parquet.unionByName(watch_parquet)

# Acción para forzar la ejecución
num_registros_parquet = final_parquet.count()

fin_parquet = time.time()

tiempo_parquet = fin_parquet - inicio_parquet

print("Número de registros:", num_registros_parquet)
print("Tiempo Parquet:", tiempo_parquet, "segundos")

Número de registros: 307
Tiempo Parquet: 10.215559244155884 segundos


### Caso 2: DataFrames creados a partir de CSV

Se mide el tiempo necesario para leer los cuatro ficheros CSV originales y
realizar exactamente el mismo procesamiento utilizado en el caso anterior.

Los ficheros CSV se leen utilizando el esquema definido previamente.

In [11]:
inicio_csv = time.time()

# Lectura de los ficheros CSV
c_p_acc = cargar_csv("Phones_accelerometer.csv")
c_p_gyro = cargar_csv("Phones_gyroscope.csv")
c_w_acc = cargar_csv("Watch_accelerometer.csv")
c_w_gyro = cargar_csv("Watch_gyroscope.csv")

# Agregación
c_p_acc_agg = agregar_movimientos(c_p_acc, "acc")
c_p_gyro_agg = agregar_movimientos(c_p_gyro, "gyro")
c_w_acc_agg = agregar_movimientos(c_w_acc, "acc")
c_w_gyro_agg = agregar_movimientos(c_w_gyro, "gyro")

# Join de teléfonos
phones_csv = c_p_acc_agg.join(
    c_p_gyro_agg,
    ["User", "Model", "gt"],
    "inner"
)

# Join de relojes
watch_csv = c_w_acc_agg.join(
    c_w_gyro_agg,
    ["User", "Model", "gt"],
    "inner"
)

# Union final
final_csv = phones_csv.unionByName(watch_csv)

# Acción para forzar la ejecución
num_registros_csv = final_csv.count()

fin_csv = time.time()

tiempo_csv = fin_csv - inicio_csv

print("Número de registros:", num_registros_csv)
print("Tiempo CSV:", tiempo_csv, "segundos")

[Stage 29:===================================================>     (9 + 1) / 10]

Número de registros: 307
Tiempo CSV: 33.08685302734375 segundos


### Tabla de tiempos de ejecución

La siguiente tabla muestra los tiempos obtenidos al ejecutar el mismo
procesamiento utilizando los ficheros Parquet y los ficheros CSV originales.

In [12]:
resultados_tiempo = [
    ("Caso 1 - Parquet", tiempo_parquet),
    ("Caso 2 - CSV", tiempo_csv)
]

tabla_tiempos = spark.createDataFrame(
    resultados_tiempo,
    ["Caso", "Tiempo_segundos"]
)

tabla_tiempos.show(truncate=False)

+----------------+------------------+
|Caso            |Tiempo_segundos   |
+----------------+------------------+
|Caso 1 - Parquet|10.215559244155884|
|Caso 2 - CSV    |33.08685302734375 |
+----------------+------------------+



## 3. Conclusiones

Los resultados obtenidos muestran que el formato Parquet reduce
considerablemente el espacio de almacenamiento respecto a los ficheros CSV
originales.

Además, el procesamiento utilizando los ficheros Parquet ha sido más rápido.
En esta ejecución, el procesamiento con Parquet ha tardado aproximadamente
10,22 segundos, mientras que utilizando los ficheros CSV ha tardado
aproximadamente 33,09 segundos.

Por tanto, para este conjunto de datos, el formato Parquet ofrece ventajas
tanto en términos de almacenamiento como de tiempo de procesamiento.